# Psoriasis scRNA-seq Drug Repurposing Pipeline
**Transcriptome-Guided Drug Repurposing for Psoriasis**

Glen Ritschel | Ritschel Research | 2026

GitHub: https://github.com/glenritschel/psoriasis-scrna

**Dataset**: GSE173706 — PP (lesional) vs PN (uninvolved psoriatic), 25 samples, CSV format.

**Strategy**: Mount Google Drive first. All intermediate files are saved to Drive
immediately after each step. If the runtime resets, rerun from the step that failed
— completed steps will reload from Drive automatically.

**Before running**: Runtime > Change runtime type > T4 GPU

## Step 1: Mount Google Drive
**Run this first.** All outputs are written directly to Drive throughout the pipeline.

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/psoriasis_scrna_output"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Drive mounted. Output directory:", DRIVE_DIR)
print("Existing files:", os.listdir(DRIVE_DIR))

## Step 2: Clone repo and install dependencies

In [ ]:
import os
REPO_URL = "https://github.com/glenritschel/psoriasis-scrna"
REPO_DIR = "psoriasis-scrna"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    !cd {REPO_DIR} && git pull
%cd /content/{REPO_DIR}/notebooks
print("Working directory:", os.getcwd())

In [ ]:
import subprocess, sys
packages = [
    "scvi-tools==1.3.3", "scanpy==1.11.5", "gseapy==1.1.10",
    "leidenalg", "python-igraph", "scikit-misc", "biopython", "pybiomart"
]
for pkg in packages:
    print("Installing", pkg, "...")
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)
print("All dependencies installed.")

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("WARNING: No GPU detected.")

## Step 3: Download raw data
Downloads GSE173706_RAW.tar (252MB) and the Ensembl ID mapping.
Both are saved to Drive and will be reused on subsequent runs.

In [ ]:
import os, urllib.request, tarfile, glob

DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/psoriasis_scrna_output"
RAW_DIR = os.path.join(DRIVE_DIR, "raw")
os.makedirs(RAW_DIR, exist_ok=True)

# Download tar to Drive
TAR_PATH = os.path.join(RAW_DIR, "GSE173706_RAW.tar")
TAR_URL = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE173nnn/GSE173706/suppl/GSE173706_RAW.tar"
if not os.path.exists(TAR_PATH):
    print("Downloading GSE173706_RAW.tar to Drive (252MB)...", flush=True)
    urllib.request.urlretrieve(TAR_URL, TAR_PATH)
    print("Done.", round(os.path.getsize(TAR_PATH)/1e6), "MB")
else:
    print("TAR already on Drive:", TAR_PATH)

# Extract to Drive
if len(glob.glob(os.path.join(RAW_DIR, "*.csv.gz"))) < 10:
    print("Extracting...")
    with tarfile.open(TAR_PATH) as tf:
        tf.extractall(RAW_DIR)
    print("Extraction complete.")

csv_files = sorted(glob.glob(os.path.join(RAW_DIR, "GSM*.csv.gz")))
print(f"CSV files: {len(csv_files)}")
for cond in ["PP", "PN", "NS"]:
    n = sum(1 for f in csv_files if f"_{cond}-" in os.path.basename(f))
    print(f"  {cond}: {n} samples")

In [ ]:
import os, pandas as pd
DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/psoriasis_scrna_output"
RAW_DIR = os.path.join(DRIVE_DIR, "raw")
MAPPING_PATH = os.path.join(RAW_DIR, "ensembl_to_symbol.csv")

if not os.path.exists(MAPPING_PATH):
    from pybiomart import Dataset
    print("Querying Ensembl BioMart...")
    ds = Dataset(name="hsapiens_gene_ensembl", host="http://www.ensembl.org")
    mapping = ds.query(attributes=["ensembl_gene_id", "hgnc_symbol"])
    mapping.to_csv(MAPPING_PATH, index=False)
    print("Saved to Drive:", len(mapping), "entries")
else:
    mapping = pd.read_csv(MAPPING_PATH)
    print("Mapping already on Drive:", len(mapping), "entries")
print(mapping.head())

## Step 4: Load, QC, and save to Drive
Loads PP and PN samples one condition at a time to manage memory.
Each batch is saved to Drive before loading the next.
If this step was completed in a previous run, it will reload from Drive.

In [ ]:
import os, re, glob, gc
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/psoriasis_scrna_output"
RAW_DIR = os.path.join(DRIVE_DIR, "raw")
MAPPING_PATH = os.path.join(RAW_DIR, "ensembl_to_symbol.csv")
QC_PATH = os.path.join(DRIVE_DIR, "adata_qc.h5ad")

QC_MIN_GENES = 200
QC_MIN_CELLS = 3
QC_MAX_MITO_PCT = 20.0
N_HVG = 4000
USE_CONDITIONS = ["PP", "PN"]

if os.path.exists(QC_PATH):
    print("QC object already on Drive. Loading...")
    adata_qc = sc.read_h5ad(QC_PATH)
    print("Loaded:", adata_qc.n_obs, "cells x", adata_qc.n_vars, "genes")
    for c in USE_CONDITIONS:
        print(" ", c + ":", (adata_qc.obs["condition"] == c).sum())
else:
    # Load mapping
    mapping_df = pd.read_csv(MAPPING_PATH)
    ensg_col = [c for c in mapping_df.columns if "stable" in c.lower() or "ensembl" in c.lower()][0]
    sym_col  = [c for c in mapping_df.columns if "symbol" in c.lower() or "hgnc" in c.lower()][0]
    ensg_to_symbol = {k: v for k, v in zip(mapping_df[ensg_col], mapping_df[sym_col])
                      if isinstance(v, str) and v.strip() != ""}
    del mapping_df; gc.collect()
    print("Mapping loaded:", len(ensg_to_symbol))

    def parse_filename(fname):
        m = re.match(r"(GSM\d+)_(NS|PN|PP)-(.+)\.csv\.gz$", fname)
        return (m.group(1), m.group(2), m.group(3)) if m else (None, None, None)

    def load_csv_sample(csv_path, sample_id, condition, patient):
        df = pd.read_csv(csv_path, index_col=0).T
        df.columns = df.columns.map(lambda g: ensg_to_symbol.get(g, g))
        df = df.loc[:, df.columns != ""]
        df = df.loc[:, ~df.columns.duplicated()]
        adata = sc.AnnData(X=sp.csr_matrix(df.values, dtype=np.float32))
        adata.obs_names = [sample_id + "_" + bc for bc in df.index]
        adata.var_names = df.columns.tolist()
        adata.var_names_make_unique()
        del df; gc.collect()
        sc.pp.filter_cells(adata, min_counts=1)
        adata.obs["sample"] = sample_id
        adata.obs["condition"] = condition
        adata.obs["patient"] = patient
        return adata

    csv_files = sorted(glob.glob(os.path.join(RAW_DIR, "GSM*.csv.gz")))
    batch_paths = []

    for cond in USE_CONDITIONS:
        batch_path = os.path.join(DRIVE_DIR, f"batch_{cond}.h5ad")
        if os.path.exists(batch_path):
            print(f"Batch {cond} already on Drive, skipping.")
            batch_paths.append(batch_path)
            continue
        print(f"\nLoading {cond} samples...")
        adatas = []
        for f in csv_files:
            gsm, condition, patient = parse_filename(os.path.basename(f))
            if condition != cond:
                continue
            sid = gsm + "_" + cond + "_" + patient
            print(f"  {sid}...", end=" ", flush=True)
            a = load_csv_sample(f, sid, cond, patient)
            adatas.append(a)
            print(a.n_obs, "cells")
        print(f"  Concatenating {cond}...")
        batch = sc.concat(adatas, join="outer", fill_value=0)
        del adatas; gc.collect()
        print(f"  {cond}: {batch.n_obs} cells — saving to Drive...")
        batch.write_h5ad(batch_path)
        del batch; gc.collect()
        print(f"  Saved: {batch_path}")
        batch_paths.append(batch_path)

    # Concatenate batches
    print("\nConcatenating batches from Drive...")
    adatas = [sc.read_h5ad(p) for p in batch_paths]
    adata = sc.concat(adatas, join="outer", fill_value=0)
    del adatas; gc.collect()
    adata.layers["counts"] = sp.csr_matrix(adata.X) if not sp.issparse(adata.X) else adata.X.copy()
    print("Combined:", adata.n_obs, "cells x", adata.n_vars, "genes")

    # QC
    print("\nRunning QC...")
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
    n_before = adata.n_obs
    sc.pp.filter_cells(adata, min_genes=QC_MIN_GENES)
    sc.pp.filter_genes(adata, min_cells=QC_MIN_CELLS)
    adata = adata[adata.obs["pct_counts_mt"] < QC_MAX_MITO_PCT].copy()
    print("QC:", n_before, "->", adata.n_obs, "cells")

    # HVG
    print("Selecting HVGs...")
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.layers["norm_log"] = adata.X.copy()
    sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor="seurat_v3",
                                layer="counts", batch_key="sample", subset=False)
    print("HVGs selected:", adata.var["highly_variable"].sum())

    # Save to Drive
    print("Saving QC object to Drive...")
    adata.write_h5ad(QC_PATH)
    adata_qc = adata
    print("Saved:", QC_PATH)
    for c in USE_CONDITIONS:
        print(" ", c + ":", (adata_qc.obs["condition"] == c).sum())

print("\nStep 4 complete.")

## Step 5: scVI Embedding + Leiden Clustering
Saves scVI object to Drive on completion.

In [ ]:
import os, gc
import numpy as np
import pandas as pd
import scanpy as sc
import scvi

DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/psoriasis_scrna_output"
SCVI_PATH = os.path.join(DRIVE_DIR, "adata_scvi.h5ad")
QC_PATH   = os.path.join(DRIVE_DIR, "adata_qc.h5ad")

SCVI_PARAMS = {"n_latent": 30, "n_layers": 2, "n_hidden": 128}
LEIDEN_RESOLUTIONS = [0.5, 0.8, 1.2]
RANDOM_SEED = 0

if os.path.exists(SCVI_PATH):
    print("scVI object already on Drive. Loading...")
    adata_scvi = sc.read_h5ad(SCVI_PATH)
    print("Loaded:", adata_scvi.n_obs, "cells,", adata_scvi.obs["leiden"].nunique(), "clusters")
else:
    if "adata_qc" not in dir():
        print("Loading QC object from Drive...")
        adata_qc = sc.read_h5ad(QC_PATH)
    adata_hvg = adata_qc[:, adata_qc.var["highly_variable"]].copy()
    print("HVG subset:", adata_hvg.n_obs, "x", adata_hvg.n_vars)

    # Train scVI
    import torch, random
    np.random.seed(RANDOM_SEED); random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
    scvi.settings.seed = RANDOM_SEED
    scvi.model.SCVI.setup_anndata(adata_hvg, layer="counts", batch_key="sample")
    model = scvi.model.SCVI(adata_hvg, **SCVI_PARAMS)
    print("Training scVI on", adata_hvg.n_obs, "cells...")
    model.train(max_epochs=400, early_stopping=False, accelerator="auto")
    final_loss = model.history["train_loss_epoch"].values[-1]
    print("Training complete. Loss:", float(np.array(final_loss).flat[0]))
    adata_hvg.obsm["X_scVI"] = model.get_latent_representation()

    # Neighbours and UMAP
    sc.pp.neighbors(adata_hvg, use_rep="X_scVI", n_neighbors=15)
    sc.tl.umap(adata_hvg)

    # Leiden multi-resolution
    for res in LEIDEN_RESOLUTIONS:
        key = "leiden_" + str(res)
        sc.tl.leiden(adata_hvg, resolution=res, key_added=key, random_state=RANDOM_SEED,
                     flavor="igraph", n_iterations=2, directed=False)
        print("Resolution", res, ":", adata_hvg.obs[key].nunique(), "clusters")

    # Auto-select best resolution (most clusters with good separation)
    adata_hvg.obs["leiden"] = adata_hvg.obs["leiden_0.8"].copy()
    print("Using resolution 0.8:", adata_hvg.obs["leiden"].nunique(), "clusters")

    print("Saving scVI object to Drive...")
    adata_hvg.write_h5ad(SCVI_PATH)
    adata_scvi = adata_hvg
    print("Saved:", SCVI_PATH)

print("\nStep 5 complete.")

## Step 6: Scripts 03-07 — Annotate, Score, DE, LINCS, Novelty

In [ ]:
# Set PROCESSED_DIR to Drive so scripts save there
import os
DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/psoriasis_scrna_output"
os.makedirs(DRIVE_DIR, exist_ok=True)

# Symlink processed dir to Drive so %run scripts save directly there
LOCAL_PROCESSED = "/content/psoriasis-scrna/data/processed"
if not os.path.exists(LOCAL_PROCESSED):
    os.makedirs("/content/psoriasis-scrna/data", exist_ok=True)
    os.symlink(DRIVE_DIR, LOCAL_PROCESSED)
    print("Symlinked processed -> Drive")
else:
    print("Processed dir exists:", LOCAL_PROCESSED)

# Copy scVI object to local processed dir if not already there
import shutil
scvi_local = os.path.join(LOCAL_PROCESSED, "adata_scvi.h5ad")
scvi_drive = os.path.join(DRIVE_DIR, "adata_scvi.h5ad")
if not os.path.exists(scvi_local) and os.path.exists(scvi_drive):
    # Already linked via symlink — nothing to do
    pass
print("scVI object accessible:", os.path.exists(scvi_local))

In [ ]:
%run ../src/03_annotate_clusters.py

In [ ]:
%run ../src/04_signature_scoring.py

In [ ]:
%run ../src/05_differential_expression.py

In [ ]:
%run ../src/06_lincs_repurposing.py

In [ ]:
%run ../src/07_novelty_prioritization.py

## Step 7: Verify outputs on Drive
All outputs should already be on Drive. This cell confirms what was saved.

In [ ]:
import os, glob
DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/psoriasis_scrna_output"
files = sorted(glob.glob(os.path.join(DRIVE_DIR, "*.csv")) +
               glob.glob(os.path.join(DRIVE_DIR, "*.h5ad")))
print(f"{len(files)} output files on Drive:")
for f in files:
    size = os.path.getsize(f)
    print(f"  {os.path.basename(f)} ({size//1024:,} KB)")